## Lab 15 – Table Clustering in Snowflake
**Snowflake Fundamentals 4-day class Lab · © 2026 Innovation In Software Corporation. All rights reserved.**

Topics covered:
1. Creating clustered vs. non-clustered tables
2. Loading 50 million rows with the GENERATOR function
3. Query performance comparison — partition pruning in action
4. EXPLAIN plan — comparing `partitionsAssigned` vs. `partitionsTotal`
5. SYSTEM$CLUSTERING_INFORMATION — average depth and partition count

### DEMO 1 | Context Setup

> **[INSTRUCTOR NOTE]**
> - Clustering is most impactful on large tables — we generate 50 million rows to make the difference measurable. 
> - Use at least an X-Small warehouse; a Small will speed up the inserts noticeably.

In [ ]:
%%sql -r dataframe_2
CREATE DATABASE IF NOT EXISTS DEMO_DB;
CREATE SCHEMA IF NOT EXISTS DEMO_DB.DEMO_SCHEMA;
USE DATABASE DEMO_DB;
USE SCHEMA DEMO_SCHEMA;
CREATE WAREHOUSE IF NOT EXISTS DEMO_WH WITH WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 300;
USE WAREHOUSE DEMO_WH;

### DEMO 2 | Create Non-Clustered Table

> **[INSTRUCTOR NOTE]**
> - A TRANSIENT table is used to avoid Fail-safe storage costs during the lab. 
> - The non-clustered table stores rows in insertion order — micro-partitions are NOT sorted by `sale_date`, so a date-range filter must scan many more partitions.

In [ ]:
%%sql -r dataframe_3
-- Non-clustered table
CREATE OR REPLACE TRANSIENT TABLE sales_non_clustered (
    sale_id      INT,
    product_name STRING,
    amount       DECIMAL(10,2),
    sale_date    DATE
);

### Populate Tables with Sample Data

> **[INSTRUCTOR NOTE]**
> - Insert 50 million rows. `SEQ4()` generates sequential integers, `RANDOM()` provides variety, and `GENERATOR(ROWCOUNT => 50000000)` drives the row count. 
> - This may take 1–3 minutes on X-Small — a good time to explain micro-partitions while it runs.

In [ ]:
INSERT INTO sales_non_clustered
    SELECT (SEQ4() % 2147483647) + 1                          AS sale_id,
           CASE WHEN RANDOM() < 0.5 THEN 'Laptop'
                ELSE 'Phone' 
           END                                                AS product_name,
           UNIFORM(0, 9999, RANDOM()) / 100.0                 AS amount,
           DATEADD('DAY', -(RANDOM() % 365), '2025-09-21')    AS sale_date
    FROM TABLE(GENERATOR(ROWCOUNT => 50000000));

### DEMO 3 | Create Clustered Table

> **[INSTRUCTOR NOTE]**
> - `CLUSTER BY (sale_date)` tells Snowflake to sort data into micro-partitions by date as it is written. 
> - Creating it as `AS SELECT * FROM sales_non_clustered` copies the 50 M rows and re-sorts them in one operation.

In [ ]:
%%sql -r dataframe_5
-- Clustered table (created from the non-clustered source, sorted by sale_date)
CREATE OR REPLACE TRANSIENT TABLE sales_clustered
CLUSTER BY (sale_date)
AS SELECT * FROM sales_non_clustered;

### DEMO 4 | Query Performance Comparison

> **[INSTRUCTOR NOTE]**
> - Run both queries and compare execution times in the Query History. 
> - The clustered table should complete in a fraction of the time because Snowflake prunes most micro-partitions based on the date range filter.

In [ ]:
%%sql -r dataframe_6
-- Non-clustered query
SELECT product_name, SUM(amount) AS total_sales
FROM sales_non_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name;

In [ ]:
 -- Clustered query
SELECT product_name, SUM(amount) AS total_sales
FROM sales_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name;

### DEMO 5 | EXPLAIN Plan — Partition Pruning

> **[INSTRUCTOR NOTE]**
> - The EXPLAIN output shows `partitionsTotal` vs. `partitionsAssigned`. 
> - For the clustered table, `partitionsAssigned` should be a small fraction of `partitionsTotal`, proving that most partitions were pruned. 
> - The non-clustered table will show nearly all partitions assigned.

In [ ]:
%%sql -r dataframe_8
EXPLAIN
SELECT product_name, SUM(amount) AS total_sales
FROM sales_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name
->>
SELECT "operation", "partitionsTotal", "partitionsAssigned", "bytesAssigned"
FROM $1
WHERE "operation" = 'TableScan';

In [ ]:
%%sql -r dataframe_9
EXPLAIN
SELECT product_name, SUM(amount) AS total_sales
FROM sales_non_clustered
WHERE sale_date BETWEEN '2025-01-01' AND '2025-01-02'
GROUP BY product_name
->>
SELECT "operation", "partitionsTotal", "partitionsAssigned", "bytesAssigned"
FROM $1
WHERE "operation" = 'TableScan';

### DEMO 6 | Clustering Information

> **[INSTRUCTOR NOTE]**
> - `SYSTEM$CLUSTERING_INFORMATION` returns JSON metadata about the clustering depth. `average_depth` close to 1 means excellent clustering — each date range maps to very few overlapping partitions. 
> - A high value on the non-clustered table shows poor clustering.

In [ ]:
%%sql -r dataframe_10
WITH x AS (
    SELECT PARSE_JSON(
        SYSTEM$CLUSTERING_INFORMATION('DEMO_DB.DEMO_SCHEMA.SALES_CLUSTERED', '(SALE_DATE)')
    ) AS meta
)
SELECT
    meta:total_partition_count AS total_partition_count,
    meta:average_depth         AS average_depth
FROM x;

In [ ]:
%%sql -r dataframe_11
WITH x AS (
    SELECT PARSE_JSON(
        SYSTEM$CLUSTERING_INFORMATION('DEMO_DB.DEMO_SCHEMA.SALES_NON_CLUSTERED', '(SALE_DATE)')
    ) AS meta
)
SELECT
    meta:total_partition_count AS total_partition_count,
    meta:average_depth         AS average_depth
FROM x;

### DEMO CLEANUP

> Drop objects created during the demo.

In [ ]:
%%sql -r dataframe_12
-- DROP DATABASE IF EXISTS DEMO_DB;   -- uncomment if you are not continuing to the next lab
-- DROP WAREHOUSE IF EXISTS DEMO_WH;  -- uncomment if you are not continuing to the next lab
USE WAREHOUSE COMPUTE_WH;